# Tel Aviv Business Data Transformation - Silver Layer

## Overview
This notebook implements the **Silver layer** transformation for Tel Aviv municipality business registry data. It parses raw JSON payloads from the bronze layer, applies data quality rules, cleanses street names, and produces a structured, analytics-ready dataset.

## Purpose
* Parse JSON payloads from bronze layer into structured columns
* Cleanse street names by removing symbols and special characters
* Handle null values and empty strings consistently
* Cast data types appropriately (dates, numbers, booleans)
* Deduplicate business records using most recent source timestamp
* Produce clean, conformed data for downstream dimensional modeling

## Key Features
* **JSON Schema Inference**: Automatically detects schema from sample payload
* **Street Name Cleansing**: Removes symbols/special characters, keeps only letters, numbers, Hebrew characters, and spaces
* **Null Handling**: Converts empty strings to NULL for consistency
* **Type Safety**: Uses `try_cast` to handle invalid data gracefully
* **Date Parsing**: Converts Unix timestamps and formatted strings to proper date types
* **Deduplication**: Window function keeps most recent record per `business_id`
* **Boolean Normalization**: Handles multiple boolean representations (Y/N, 1/0)

## Data Source
**Input**: `workspace.tel_aviv_municipality_raw.business`

## Output Table
**Target**: `workspace.tel_aviv_municipality_stg.business` 

**Schema**:
| Column | Type | Description |
|--------|------|-------------|
| `_event_id` | STRING | Lineage: original event ID from bronze |
| `_uuid` | STRING | Lineage: original record UUID from bronze |
| `_ingestion_at` | TIMESTAMP | Lineage: original ingestion timestamp |
| `object_id` | LONG | ArcGIS object identifier |
| `business_id` | LONG | Unique business identifier (primary key) |
| `primary_holder_id` | LONG | Primary license holder identifier |
| `primary_holder_name` | STRING | Name of primary license holder |
| `foundation_date` | DATE | Business foundation/establishment date |
| `street_name` | STRING | **Cleansed** street name (symbols removed) |
| `house_number` | INT | Street address number |
| `type` | STRING | Business type/category |
| `is_licensing_required` | BOOLEAN | Whether licensing is required |
| `x_coordinate` | DOUBLE | Geographic X coordinate |
| `y_coordinate` | DOUBLE | Geographic Y coordinate |
| `house_area_sqm` | DOUBLE | Property area in square meters |
| `source_updated_at` | TIMESTAMP | Last update timestamp from source system |

## Data Cleansing Rules

### Street Name Cleansing
**Function**: `clean_signals_and_nulls(col_name)`
* **Removes**: All symbols and special characters
* **Keeps**: English letters (a-z, A-Z), numbers (0-9), Hebrew characters (א-ת), spaces
* **Pattern**: `[^a-zA-Z0-9א-ת\s]` (regex)
* **Null Handling**: Empty strings after cleaning become NULL

### Null Value Handling
* Empty strings (`""`) → NULL
* Whitespace-only strings → NULL (after trim)
* Invalid casts → NULL (via `try_cast`)

### Boolean Conversion
`is_licensing_required` accepts:
* `"Y"` or `"1"` → `TRUE`
* `"N"` or `"0"` → `FALSE`
* Any other value → `NULL`

### Date Parsing
* **Unix Timestamp**: `tr_hakama` (milliseconds) → `foundation_date`
* **Formatted String**: `date_import` (dd/MM/yyyy HH:mm:ss) → `source_updated_at`

## Deduplication Strategy
**Method**: Window function with `row_number()`
* **Partition By**: `business_id`
* **Order By**: `source_updated_at DESC` (most recent first)
* **Keep**: Row 1 (most recent record per business)
* **Filter**: Removes records where `business_id IS NULL`

**Rationale**: If multiple records exist for the same business (due to API updates or reruns), keep only the most recently updated version from the source system.

## Processing Mode
**Current Mode**: Full overwrite
* Replaces entire silver table on each run
* Schema evolution enabled with `overwriteSchema = true`
* Suitable for small-to-medium datasets where full refresh is acceptable

## Design Principles
* **Schema-on-Write**: Parse JSON and enforce schema at silver layer
* **Data Quality**: Consistent null handling and type casting
* **Lineage Preservation**: Maintains bronze layer audit columns (`_event_id`, `_uuid`, `_ingestion_at`)
* **Unicode Support**: Handles Hebrew characters in street names
* **Idempotency**: Produces same output given same input (deterministic deduplication)


## Integration Notes
* **Upstream**: Depends on bronze layer (`workspace.tel_aviv_municipality_raw.business`)
* **Downstream**: Feeds gold layer dimensional model (`dim_business`)
* **Joins**: `street_name` column used to join with street closures data
* **SCD Type 2**: Gold layer tracks historical changes using this silver data

In [0]:
import json
import requests
import uuid
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, DateType
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable

In [0]:
# Helper function for strings
def null_if_empty(col_name):
    """
    Cleans a string column by trimming whitespace and converting empty strings to NULL.
    
    This is a critical utility for Silver-layer normalization. It ensures that 
    streets or business names containing only spaces or empty quotes do not 
    break joins or appear as valid data in the Gold layer.

    Args:
        col_name (str): The name of the column to process.

    Returns:
        Column: A Spark SQL Column object that is trimmed and nullified if empty.
    """
    return F.when(F.trim(F.col(col_name)) == "", None).otherwise(F.trim(F.col(col_name)))

In [0]:
def clean_signals_and_nulls(col_name):
    """
    Standardizes text columns by removing special characters and handling empty results.
    
    This function performs three key operations:
    1. Trims leading/trailing whitespace.
    2. Uses Regex to strip out 'signals' (punctuation, symbols, non-standard chars),
       keeping only Hebrew (א-ת), English (a-z), Numbers, and Spaces.
    3. Converts any resulting empty strings to NULL to maintain data integrity.

    Args:
        col_name (str): The name of the column to clean.

    Returns:
        Column: A sanitized Spark SQL Column.
    """
    # Regex: Keeps only English letters, Numbers, Hebrew (א-ת), and Spaces
    clean_pattern = r"[^a-zA-Z0-9א-ת\s]"
    raw_col = F.trim(F.col(col_name))
    
    # Replace symbols with empty string
    cleaned = F.regexp_replace(raw_col, clean_pattern, "")
    
    # If the result is empty string, return NULL
    return F.when(cleaned == "", F.lit(None)).otherwise(cleaned)

# Get Schema
sample_payload = spark.table("workspace.tel_aviv_municipality_raw.business").select("payload").first()[0]
json_schema = F.schema_of_json(sample_payload)

# Parse and Clean
silver_parsed = spark.table("workspace.tel_aviv_municipality_raw.business") \
    .withColumn("data", F.from_json(F.col("payload"), json_schema)) \
    .select(
        F.col("event_id").cast("string").alias("_event_id"),
        F.col("uuid").cast("string").alias("_uuid"),
        F.col("ingestion_at").alias("_ingestion_at"), 
        
        F.expr("try_cast(data.OBJECTID as long)").alias("object_id"),
        F.expr("try_cast(data.id_esek as long)").alias("business_id"),
        F.expr("try_cast(data.ms_mezahe_machzik_rashi as long)").alias("primary_holder_id"),
        # Applying null_if_empty (assuming you have this defined, or use F.when)
        F.when(F.trim(F.col("data.shem_machzik_rashi")) == "", F.lit(None))
         .otherwise(F.col("data.shem_machzik_rashi")).alias("primary_holder_name"),

        F.from_unixtime(F.expr("try_cast(data.tr_hakama as double) / 1000")).cast("date").alias("foundation_date"),

        # APPLIED SIGNAL REMOVAL TO STREET NAME
        clean_signals_and_nulls("data.shem_rechov").alias("street_name"),

        F.expr("try_cast(data.ms_bayit as int)").alias("house_number"),
        F.when(F.trim(F.col("data.shimush")) == "", F.lit(None)).otherwise(F.col("data.shimush")).alias("type"),
        
        F.when(
            (F.col("data.sw_taun_rishui").cast("string") == "Y") | 
            (F.col("data.sw_taun_rishui").cast("string") == "1"), 
            True
        ).when(
            (F.col("data.sw_taun_rishui").cast("string") == "N") | 
            (F.col("data.sw_taun_rishui").cast("string") == "0"),
            False
        ).otherwise(None).alias("is_licensing_required"),
        
        F.expr("try_cast(data.x_coord as double)").alias("x_coordinate"),
        F.expr("try_cast(data.y_coord as double)").alias("y_coordinate"),
        F.expr("try_cast(data.shetach as double)").alias("house_area_sqm"),

        F.to_timestamp(F.col("data.date_import"), "dd/MM/yyyy HH:mm:ss").alias("source_updated_at")
    )

# Deduplicate (Business logic: keeping most recent ingestion)
window_spec = Window.partitionBy("business_id").orderBy(F.col("source_updated_at").desc())

silver_business_final = silver_parsed \
    .filter(F.col("business_id").isNotNull()) \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

silver_business_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.tel_aviv_municipality_stg.business")